# End-to-End Apartment Price Prediction 🏠
**Портфолио-проект:** полный цикл ML — от сырых данных до анализа ошибок и сравнения моделей.

**Задача (regression):** предсказать цену квартиры по характеристикам.
**Стек:** pandas, scikit-learn (Pipeline, ColumnTransformer), matplotlib.

## Этапы
1. Данные (генерация + сохранение в data/) → 2. EDA → 3. Preprocessing (числовые + категориальные)
→ 4. Split → 5. Обучение 3 моделей → 6. Метрики и сравнение → 7. Error analysis → 8. Выводы

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
pd.set_option('display.width', 120)

In [ ]:
# ===== 1. ДАННЫЕ =====
# Синтетика, похожая на реальный рынок: нелинейности, категории, пропуски, выбросы
rng = np.random.default_rng(2024)
n = 1200

district = rng.choice(["центр", "спальный", "окраина"], n, p=[0.25, 0.5, 0.25])
district_mult = pd.Series(district).map({"центр": 1.5, "спальный": 1.0, "окраина": 0.75}).values

df = pd.DataFrame({
    "площадь_м2":   rng.uniform(28, 160, n).round(0),
    "комнаты":      rng.integers(1, 5, n),
    "этаж":         rng.integers(1, 20, n),
    "возраст_дома": rng.integers(0, 55, n),
    "до_метро_мин": rng.uniform(1, 40, n).round(0),
    "район":        district,
})
base = (2.2 * df["площадь_м2"]
        + 14 * df["комнаты"]
        - 0.7 * df["возраст_дома"]
        - 1.1 * df["до_метро_мин"]
        + 60)
df["цена_тыс"] = (base * district_mult + rng.normal(0, 22, n)).round(1)

# Реализм: 2% пропусков в возрасте + 8 квартир-выбросов ("элитки")
df.loc[rng.choice(n, int(n*0.02), replace=False), "возраст_дома"] = np.nan
lux = rng.choice(n, 8, replace=False)
df.loc[lux, "цена_тыс"] *= 2.5

df.to_csv("data_apartments.csv", index=False)   # в репозитории лежит в data/
print(df.shape); df.head()

In [ ]:
# ===== 2. EDA =====
print(df.describe().round(1))
print("\nПропуски:\n", df.isna().sum())
print("\nКорреляция числовых признаков с ценой:")
print(df.select_dtypes("number").corr()["цена_тыс"].drop("цена_тыс").round(2).sort_values())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(df["площадь_м2"], df["цена_тыс"], s=8); axes[0].set_xlabel("площадь"); axes[0].set_ylabel("цена")
df.boxplot(column="цена_тыс", by="район", ax=axes[1])
axes[2].hist(df["цена_тыс"], bins=40); axes[2].set_title("Распределение цены (виден хвост-выбросы)")
plt.tight_layout(); plt.show()

### Наблюдения EDA
- Площадь — сильнейшая линейная связь с ценой; до_метро и возраст — отрицательные.
- Район даёт сдвиг уровня цен (boxplot): «центр» заметно дороже — категорию НУЖНО кодировать, выбрасывать нельзя.
- В хвосте распределения ~8 аномально дорогих квартир — кандидаты на отдельную обработку.
- 2% пропусков в возрасте — заполним медианой внутри pipeline (без утечки из test).

In [ ]:
# ===== 3-4. PREPROCESSING + SPLIT =====
from sklearn.impute import SimpleImputer

X = df.drop(columns="цена_тыс")
y = df["цена_тыс"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ["площадь_м2", "комнаты", "этаж", "возраст_дома", "до_метро_мин"]
cat_cols = ["район"]

# ColumnTransformer: числовые -> impute+scale, категориальные -> one-hot.
# Всё внутри pipeline => fit только на train, утечки нет.
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

In [ ]:
# ===== 5-6. ТРИ МОДЕЛИ И СРАВНЕНИЕ =====
models = {
    "LinearRegression": LinearRegression(),
    "Ridge(alpha=10)":  Ridge(alpha=10),
    "RandomForest":     RandomForestRegressor(n_estimators=300, random_state=0),
}
results = {}
for name, m in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", m)]).fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    cv = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring="r2")
    results[name] = {"pipe": pipe, "pred": pred,
                     "MAE": mean_absolute_error(y_te, pred),
                     "R2": r2_score(y_te, pred),
                     "CV R2 (mean±std)": f"{cv.mean():.3f} ± {cv.std():.3f}"}

summary = pd.DataFrame({k: {"MAE тыс.": round(v["MAE"], 1), "R2 test": round(v["R2"], 3),
                            "CV R2": v["CV R2 (mean±std)"]} for k, v in results.items()}).T
print(summary)

In [ ]:
# ===== 7. ERROR ANALYSIS (по лучшей модели) =====
best_name = max(results, key=lambda k: results[k]["R2"])
best = results[best_name]
print("Лучшая модель:", best_name)

res = y_te - best["pred"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_te, best["pred"], alpha=0.5, s=10)
axes[0].plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--')
axes[0].set_xlabel("реальная"); axes[0].set_ylabel("предсказанная"); axes[0].set_title("Predicted vs Actual")
axes[1].hist(res, bins=30); axes[1].set_title("Остатки")
axes[2].scatter(best["pred"], res, alpha=0.5, s=10); axes[2].axhline(0, c='r', ls='--')
axes[2].set_xlabel("предсказание"); axes[2].set_title("Остатки vs предсказание")
plt.tight_layout(); plt.show()

# Топ-10 худших промахов: КТО они?
worst_idx = res.abs().sort_values(ascending=False).head(10).index
worst = X_te.loc[worst_idx].copy()
worst["реальная"] = y_te.loc[worst_idx]
worst["предсказание"] = best["pred"][[list(X_te.index).index(i) for i in worst_idx]].round(1)
worst["ошибка"] = (worst["реальная"] - worst["предсказание"]).round(1)
print(worst.sort_values("ошибка"))

### Разбор ошибок
- Почти все худшие промахи — те самые «элитные» квартиры с ценой ×2.5: модель не знает признака
  «элитность» и систематически недооценивает их. Это вывод про ДАННЫЕ, а не про модель:
  нужен признак (ремонт/класс жилья) либо отдельная модель для люкс-сегмента.
- На обычных квартирах остатки симметричны, без узора → систематических ошибок нет.
- RandomForest обычно выигрывает у линейных за счёт нелинейности «район × площадь».

## ===== 8. ВЫВОДЫ =====
1. Полный пайплайн: импутация + scaling + one-hot внутри Pipeline — ноль утечек, готово к продакшену.
2. Сравнение по test и 5-fold CV: устойчивость результата проверена, а не «повезло на одном split».
3. Error analysis нашёл сегмент, где модель слепа, и подсказал, какие данные собирать дальше.

## Идеи улучшения до production-level
- Признаки: год ремонта, класс ЖК, гео-координаты, фото (CV!).
- Подбор гиперпараметров: GridSearchCV/Optuna по RF и Ridge.
- log-трансформация цены против скошенного распределения.
- Отдельная обработка/модель люкс-сегмента; quantile regression для интервалов цены.
- Обёртка в FastAPI-сервис + мониторинг дрейфа данных.

---
## Задания для практики 💪
1. Добавь признак «ремонт» (категория: нет/косметический/евро) в generate_data и в пайплайн. Вырос ли R²?
2. Примени np.log1p к цене перед обучением (и np.expm1 к предсказаниям). Помогло ли с выбросами-«элитками»?
3. Сделай GridSearchCV по n_estimators и max_depth для RandomForest. Насколько улучшился MAE?
4. Убери район из признаков и переобучи. Сколько R² «стоит» этот один признак?

## 5 проверочных вопросов ❓
1. Зачем импутация и scaling находятся ВНУТРИ Pipeline, а не делаются до split?
2. Что делает ColumnTransformer и почему числовые и категориальные признаки обрабатываются по-разному?
3. Чем вывод cross_val_score надёжнее одной метрики на test?
4. Почему модель систематически недооценивает «элитки» и почему это проблема данных, а не модели?
5. Какие два графика из error analysis ты покажешь заказчику и что по ним скажешь?

## Чек-лист перед выкладкой на GitHub ✅
- [ ] Прогнал весь ноутбук сверху вниз без ошибок (Restart & Run All)
- [ ] Переписал выводы своими словами
- [ ] Проверил, что python src/generate_data.py && python src/train.py && python src/predict.py работают
- [ ] Заполнил description и topics из README